#### BTS 콘서트 이후 경제파급효과 분석

#### 콘서트 기간 전후 유입인구 현황
* 시간대 성별 유입인구
* 시간대 연령 유입인구
* 성별, 연령에서 60대 이상은 제외

In [1]:
import os
import pandas as pd
from datetime import datetime as dt

In [62]:
print(" ========== 행정동별 유입인구 현황집계 시작 ========== ")
# 유입인구 행정동명을 가져오기 위한 행정동코드데이터 불러오기
hcode_data1 = pd.read_csv("./공통코드/행정동코드.csv", sep=",", encoding="utf-8")
hcode_data1['hcode'] = hcode_data1['hcode'].astype(str)
hcode_data1['admdong_cd'] = [hcode_data1.loc[i, 'hcode'][:8] for i in hcode_data1.index]
scode_data1 = pd.read_csv("./공통코드/행안부_구군코드.csv", sep=",", encoding="utf-8")
scode_data1['SIG_CD'] = scode_data1['SIG_CD'].astype(str)
del scode_data1['SIG_ENG_NM']
scode_data1.columns=['rsdn_admdong_cd', 'sname']

# 유입인구 데이터를 데이터프레임으로 변환
df1 = pd.read_csv("./본청_대변인실/data/생활인구/내국인/native_ingrspopul_20220916_20221020.csv", sep=",", encoding="utf-8")
df1['base_date'] = df1['base_date'].astype(str)
df1['base_year'] = [df1.loc[i,'base_date'][:4] for i in df1.index]
df1['base_month'] = [df1.loc[i,'base_date'][4:6] for i in df1.index]
df1['base_day'] = [df1.loc[i,'base_date'][6:] for i in df1.index]
df1['date'] = df1['base_year'] + '-' + df1['base_month'] + '-' + df1['base_day']
df1['date'] = pd.to_datetime(df1['date'])
df1['weekday'] = df1['date'].dt.weekday
df1['요일'] = df1['date'].dt.day_name()
df1['timezn_cd'] = df1['timezn_cd'].astype(str)
df1['admdong_cd'] = df1['admdong_cd'].astype(str)
df1['rsdn_admdong_cd'] = df1['rsdn_admdong_cd'].astype(str)

# 주단위 분류
df1.loc[(df1['base_month'].str.contains('09')) & (df1['base_day'].str.contains('19|20|21|22|23')), "week_num"] = "9월 3주중"
df1.loc[(df1['base_month'].str.contains('09')) & (df1['base_day'].str.contains('26|27|28|29|30')), "week_num"] = "9월 4주중"
df1.loc[(df1['base_month'].str.contains('09')) & (df1['base_day'].str.contains('17|18')), "week_num"] = "9월 2주말"
df1.loc[(df1['base_month'].str.contains('09')) & (df1['base_day'].str.contains('24|25')), "week_num"] = "9월 3주말"
df1.loc[(df1['base_month'].str.contains('10')) & (df1['base_day'].str.contains('1|2')), "week_num"] = "9월 4주말"
df1.loc[(df1['base_month'].str.contains('10')) & (df1['base_day'].str.contains('3|4|5|6|7')), "week_num"] = "10월 1주중"
df1.loc[(df1['base_month'].str.contains('10')) & (df1['base_day'].str.contains('8|9')), "week_num"] = "10월 1주말"
df1.loc[(df1['base_month'].str.contains('10')) & (df1['base_day'].str.contains('10|11|12|13|14')), "week_num"] = "10월 2주중"
df1.loc[(df1['base_month'].str.contains('10')) & (df1['base_day'].str.contains('15|16')), "week_num"] = "10월 2주말"
df1.loc[(df1['base_month'].str.contains('10')) & (df1['base_day'].str.contains('17|18|19|20')), "week_num"] = "10월 3주중"

# 콘서트 장소(행정동) 추출, 행정동코드 기준으로 행정동코드 데이터와 조인
df1 = df1.loc[df1['admdong_cd'].str.contains('26470620')].copy()
df1 = pd.merge(df1, hcode_data1, "inner", "admdong_cd")
df1 = pd.merge(df1, scode_data1, "inner", "rsdn_admdong_cd")

print(" ===== 유입인구 성별 집계시작 ===== ")
df1['sum_m'] = df1['m10'] + df1['m15'] + df1['m20'] + df1['m25'] + df1['m30'] + df1['m35'] + df1['m40'] + df1['m45'] + df1['m50'] + df1['m55']
df1['sum_f'] = df1['f10'] + df1['f15'] + df1['f20'] + df1['f25'] + df1['f30'] + df1['f35'] + df1['f40'] + df1['f45'] + df1['f50'] + df1['f55']
df1_mf_agg = df1.groupby(['date','요일','week_num','timezn_cd','admdong_cd','hname', 'rsdn_admdong_cd', 'sname'])['sum_m', 'sum_f'].sum().reset_index().sort_values(by="dcode")
df1_mf_agg.columns = ['기준일자', '요일', '주차', '시간대', '생활지코드', '생활지명', '거주지구군코드','거주지구군명','남성(합)', '여성(합)']
df1_mf_agg.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_성별유입생활인구현황.csv", sep="|", index=False, encoding="utf-8")
print(" ===== 유입인구 성별 집계완료 ===== ")
print()
print(" ===== 유입인구 연령 집계시작 ===== ")
df1['sum_10'] = df1['m10'] + df1['m15'] + df1['f10'] + df1['f15']
df1['sum_20'] = df1['m20'] + df1['m25'] + df1['f20'] + df1['f25']
df1['sum_30'] = df1['m30'] + df1['m35'] + df1['f30'] + df1['f35']
df1['sum_40'] = df1['m40'] + df1['m45'] + df1['f40'] + df1['f45']
df1['sum_50'] = df1['m50'] + df1['m55'] + df1['f50'] + df1['f55']
df1_age_agg = df1.groupby(['date','요일','week_num','timezn_cd','admdong_cd','hname','rsdn_admdong_cd', 'sname'])['sum_10', 'sum_20', 'sum_30', 'sum_40', 'sum_50'].sum().reset_index().sort_values(by="dcode")
df1_age_agg.columns=['기준일자', '요일', '주차', '시간대', '생활지코드', '생활지명', '거주지구군코드','거주지구군명', '10대(합)', '20대(합)', '30대(합)', '40대(합)', '50대(합)']
df1_age_agg.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_연령유입생활인구현황.csv", sep="|", index=False, encoding="utf-8")
print(" ===== 유입인구 연령 집계완료 ===== ")
print()
print(" ========== 행정동별 유입인구 현황집계 완료 ========== ")

 ========== 행정동별 유입인구 현황집계 시작 ========== 
 ===== 유입인구 성별 집계시작 ===== 


KeyError: 'sname'

In [ ]:
print(" ========== 행정동별 유출인구 현황집계 시작 ========== ")
# 유출인구 행정동명을 가져오기 위한 행정동코드데이터 불러오기
hcode_data2 = pd.read_csv("./공통코드/행정동코드.csv", sep=",", encoding="utf-8")
hcode_data2['hcode'] = hcode_data2['hcode'].astype(str)
hcode_data2['rsdn_admdong_cd'] = [hcode_data2.loc[i, 'hcode'][:8] for i in hcode_data2.index]
scode_data2 = pd.read_csv("./공통코드/행안부_구군코드.csv", sep=",", encoding="utf-8")
scode_data2['SIG_CD'] = scode_data2['SIG_CD'].astype(str)
del scode_data2['SIG_ENG_NM']
scode_data2.columns=['admdong_cd', 'sname']

# 유출인구 데이터프레임으로 변환
df2 = pd.read_csv("./본청_대변인실/data/생활인구/내국인/native_egrspopul_20220916_20221020.csv", sep=",", encoding="utf-8")
df2['base_date'] = df2['base_date'].astype(str)
df2['base_year'] = [df2.loc[i,'base_date'][:4] for i in df2.index]
df2['base_month'] = [df2.loc[i,'base_date'][4:6] for i in df2.index]
df2['base_day'] = [df2.loc[i,'base_date'][6:] for i in df2.index]
df2['date'] = df2['base_year'] + '-' + df2['base_month'] + '-' + df2['base_day']
df2['date'] = pd.to_datetime(df2['date'])
df2['weekday'] = df2['date'].dt.weekday
df2['요일'] = df2['date'].dt.day_name()
df2['timezn_cd'] = df2['timezn_cd'].astype(str)
df2['rsdn_admdong_cd'] = df2['rsdn_admdong_cd'].astype(str)
df2['admdong_cd'] = df2['admdong_cd'].astype(str)

# 주단위 분류
df2.loc[(df2['base_month'].str.contains('09')) & (df2['base_day'].str.contains('19|20|21|22|23')), "week_num"] = "9월 3주중"
df2.loc[(df2['base_month'].str.contains('09')) & (df2['base_day'].str.contains('26|27|28|29|30')), "week_num"] = "9월 4주중"
df2.loc[(df2['base_month'].str.contains('09')) & (df2['base_day'].str.contains('17|18')), "week_num"] = "9월 2주말"
df2.loc[(df2['base_month'].str.contains('09')) & (df2['base_day'].str.contains('24|25')), "week_num"] = "9월 3주말"
df2.loc[(df2['base_month'].str.contains('10')) & (df2['base_day'].str.contains('1|2')), "week_num"] = "9월 4주말"
df2.loc[(df2['base_month'].str.contains('10')) & (df2['base_day'].str.contains('3|4|5|6|7')), "week_num"] = "10월 1주중"
df2.loc[(df2['base_month'].str.contains('10')) & (df2['base_day'].str.contains('8|9')), "week_num"] = "10월 1주말"
df2.loc[(df2['base_month'].str.contains('10')) & (df2['base_day'].str.contains('10|11|12|13|14')), "week_num"] = "10월 2주중"
df2.loc[(df2['base_month'].str.contains('10')) & (df2['base_day'].str.contains('15|16')), "week_num"] = "10월 2주말"
df2.loc[(df2['base_month'].str.contains('10')) & (df2['base_day'].str.contains('17|18|19|20')), "week_num"] = "10월 3주중"

# 콘서트 장소(행정동) 추출, 행정동코드 기준으로 행정동코드 데이터와 조인
df2 = df2.loc[df2['rsdn_admdong_cd'].str.contains('26470620')].copy()
df2 = pd.merge(df2, hcode_data2, "inner", "rsdn_admdong_cd")
df2 = pd.merge(df2, scode_data2, "inner", "admdong_cd")

print(" ===== 유출인구 성별 집계시작 ===== ")
df2['sum_m'] = df2['m10'] + df2['m15'] + df2['m20'] + df2['m25'] + df2['m30'] + df2['m35'] + df2['m40'] + df2['m45'] + df2['m50'] + df2['m55']
df2['sum_f'] = df2['f10'] + df2['f15'] + df2['f20'] + df2['f25'] + df2['f30'] + df2['f35'] + df2['f40'] + df2['f45'] + df2['f50'] + df2['f55']
df2_mf_agg = df2.groupby(['date','요일','week_num','timezn_cd', 'rsdn_admdong_cd','hname','rsdn_admdong_cd', 'sname'])['sum_m', 'sum_f'].sum().reset_index().sort_values(by="admdong_cd")
df2_mf_agg.columns = ['기준일자', '요일', '주차', '시간대', '거주지코드', '거주지명', '생활지구군코드','생활지구군명', '남성(합)', '여성(합)']
df2_mf_agg.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_성별유출생활인구현황.csv", sep="|", index=False, encoding="utf-8")
print(" ===== 유출인구 성별 집계완료 ===== ")
print()
print(" ===== 유출인구 연령 집계시작 ===== ")
df2['sum_10'] = df2['m10'] + df2['m15'] + df2['f10'] + df2['f15']
df2['sum_20'] = df2['m20'] + df2['m25'] + df2['f20'] + df2['f25']
df2['sum_30'] = df2['m30'] + df2['m35'] + df2['f30'] + df2['f35']
df2['sum_40'] = df2['m40'] + df2['m45'] + df2['f40'] + df2['f45']
df2['sum_50'] = df2['m50'] + df2['m55'] + df2['f50'] + df2['f55']
df2_age_agg = df2.groupby(['base_month','base_day','week_num','timezn_cd','rsdn_admdong_cd','hname', 'rsdn_admdong_cd', 'sname'])['sum_10', 'sum_20', 'sum_30', 'sum_40', 'sum_50'].sum().reset_index().sort_values(by="admdong_cd")
df2_age_agg.columns=['기준월', '기준일', '주차', '시간대', '거주지코드', '거주지명', '생활지구군코드','생활지구군명', '10대(합)', '20대(합)', '30대(합)', '40대(합)', '50대(합)']
df2_age_agg.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_연령유출생활인구현황.csv", sep="|", index=False, encoding="utf-8")
print(" ===== 유출인구 연령 집계완료 ===== ")
print()
print(" ========== 행정동별 유출인구 현황집계 완료 ========== ")

#### 콘서트 기간 전후 국적별 외국인 거주현황

In [51]:

print(" ========== 행정동별 장기체류 외국인 생활인구 현황집계 시작 ========== ")
# 장기체류 외국인 현황
df3 = pd.read_csv("./본청_대변인실/data/생활인구/외국인/long_forn_admdong_sum_20220916_20221020.csv", sep=",", encoding="utf-8")
df3['base_date'] = df3['base_date'].astype(str)
df3['base_year'] = [df3.loc[i,'base_date'][:4] for i in df3.index]
df3['base_month'] = [df3.loc[i,'base_date'][4:6] for i in df3.index]
df3['base_day'] = [df3.loc[i,'base_date'][6:] for i in df3.index]
df3['date'] = df3['base_year'] + '-' + df3['base_month'] + '-' + df3['base_day']
df3['date'] = pd.to_datetime(df3['date'])
df3['weekday'] = df3['date'].dt.weekday
df3['요일'] = df3['date'].dt.day_name()
df3['timezn_cd'] = df3['timezn_cd'].astype(str)
df3['admdong_cd'] = df3['admdong_cd'].astype(str)

hcode_data1 = pd.read_csv("./공통코드/행정동코드.csv", sep=",", encoding="utf-8")
hcode_data1['hcode'] = hcode_data1['hcode'].astype(str)
hcode_data1['admdong_cd'] = [hcode_data1.loc[i, 'hcode'][:8] for i in hcode_data1.index]

df3 = df3.loc[df3['admdong_cd'].str.contains('26470620')].copy()
df3 = pd.merge(df3, hcode_data1, "inner", "admdong_cd")

# 주단위 분류
df3.loc[(df3['base_month'].str.contains('09')) & (df3['base_day'].str.contains('19|20|21|22|23')), "week_num"] = "9월 3주중"
df3.loc[(df3['base_month'].str.contains('09')) & (df3['base_day'].str.contains('26|27|28|29|30')), "week_num"] = "9월 4주중"
df3.loc[(df3['base_month'].str.contains('09')) & (df3['base_day'].str.contains('17|18')), "week_num"] = "9월 2주말"
df3.loc[(df3['base_month'].str.contains('09')) & (df3['base_day'].str.contains('24|25')), "week_num"] = "9월 3주말"
df3.loc[(df3['base_month'].str.contains('10')) & (df3['base_day'].str.contains('1|2')), "week_num"] = "9월 4주말"
df3.loc[(df3['base_month'].str.contains('10')) & (df3['base_day'].str.contains('3|4|5|6|7')), "week_num"] = "10월 1주중"
df3.loc[(df3['base_month'].str.contains('10')) & (df3['base_day'].str.contains('8|9')), "week_num"] = "10월 1주말"
df3.loc[(df3['base_month'].str.contains('10')) & (df3['base_day'].str.contains('10|11|12|13|14')), "week_num"] = "10월 2주중"
df3.loc[(df3['base_month'].str.contains('10')) & (df3['base_day'].str.contains('15|16')), "week_num"] = "10월 2주말"
df3.loc[(df3['base_month'].str.contains('10')) & (df3['base_day'].str.contains('17|18|19|20')), "week_num"] = "10월 3주중"

# 지역 기준 국적별 장기체류 생활인구 수
# 동북아시아 : 대만, 몽골, 일본, 중국
df3_agg_eastnorth = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['twn','jpn','cnh','mng'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df3_agg_eastnorth.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','타이완','일본','중국','몽골']
df3_agg_eastnorth.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_동북아시아.csv", sep="|", encoding="utf-8")

# 동남아시아 : 말레이시아, 미얀마, 베트남, 캄보디아, 태국, 필리핀, 인도네시아, 말레이시아
df3_agg_eastsouth = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['vnm','khm','phl','idn','tha','mmr','mys'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df3_agg_eastsouth.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','베트남','캄보디아','필리핀','인도네시아','태국','미얀마','말레이시아']
df3_agg_eastsouth.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_동남아시아.csv", sep="|", encoding="utf-8")

# 서남아시아 : 네팔, 스리랑카, 방글라데시, 파키스탄, 인도, 오스트레일리아
df3_agg_westsouth = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['npl','lka','bgd','pak','ind','aus','nzl'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df3_agg_westsouth.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','네팔','스리랑카','방글라데시','파키스탄','인도','호주','뉴질랜드']
df3_agg_westsouth.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_서남아시아.csv", sep="|", encoding="utf-8")

# 북미 : 미국, 캐나다
df3_agg_northameria = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['usa','can'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df3_agg_northameria.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','미국','캐나다']
df3_agg_northameria.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_북미.csv", sep="|", encoding="utf-8")

# 러시아 중앙아시아 : 러시아, 카자흐스탄, 키르기스스탄, 유즈베키스탄
df3_agg_midasia = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['rus','kaz','kgz', 'uzb'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df3_agg_midasia.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','러시아','카즈흐스탄','키르기스스탄','우즈베키스탄']
df3_agg_midasia.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_러시아중앙아시아.csv", sep="|", encoding="utf-8")

# 유럽 : 독일, 영국, 프랑스, 우크라이나
df3_agg_europe = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['gbr','deu','fra','ukr'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df3_agg_europe.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','영국','독일','프랑스','우크라이나']
df3_agg_europe.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_러시아중앙아시아.csv", sep="|", encoding="utf-8")

# 중동북아프리카 : 이집트
df3_agg_mideast = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['egy'].sum().reset_index().sort_values(by='admdong_cd')
df3_agg_mideast.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','이집트']
df3_agg_mideast.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_중동북아프리카.csv", sep="|", encoding="utf-8")

# 사하라 이남 아프리카와 기타 : 나이지리아, 기타
df3_agg_sahara = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['nga','etc'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df3_agg_sahara.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','나이지리아','기타국가']
df3_agg_sahara.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인장기체류생활인구현황_사하라이남아프리카기타.csv", sep="|", encoding="utf-8")

print(" ========== 행정동별 장기체류 외국인 생활인구 현황집계 완료 ========== ")


 ========== 행정동별 장기체류 외국인 생활인구 현황집계 시작 ========== 
 ========== 행정동별 장기체류 외국인 생활인구 현황집계 완료 ========== 


<ipython-input-51-58c5b05ade00>:36: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df3_agg_eastnorth = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['twn','jpn','cnh','mng'].sum().round(0).reset_index().sort_values(by='admdong_cd')
<ipython-input-51-58c5b05ade00>:41: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df3_agg_eastsouth = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['vnm','khm','phl','idn','tha','mmr','mys'].sum().round(0).reset_index().sort_values(by='admdong_cd')
<ipython-input-51-58c5b05ade00>:46: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df3_agg_westsouth = df3.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['npl','lka','bgd','pak','ind','aus','nzl'].sum().round

In [52]:

print(" ========== 행정동별 단기체류 외국인 생활인구 현황집계 시작 ========== ")
# 단기체류 외국인 현황
df4 = pd.read_csv("./본청_대변인실/data/생활인구/외국인/shrt_forn_admdong_sum_20220916_20221020.csv", sep=",", encoding="utf-8")
df4['base_date'] = df4['base_date'].astype(str)
df4['base_year'] = [df4.loc[i,'base_date'][:4] for i in df4.index]
df4['base_month'] = [df4.loc[i,'base_date'][4:6] for i in df4.index]
df4['base_day'] = [df4.loc[i,'base_date'][6:] for i in df4.index]
df4['date'] = df4['base_year'] + '-' + df4['base_month'] + '-' + df4['base_day']
df4['date'] = pd.to_datetime(df4['date'])
df4['weekday'] = df4['date'].dt.weekday
df4['요일'] = df4['date'].dt.day_name()
df4['timezn_cd'] = df4['timezn_cd'].astype(str)
df4['admdong_cd'] = df4['admdong_cd'].astype(str)

hcode_data1 = pd.read_csv("./공통코드/행정동코드.csv", sep=",", encoding="utf-8")
hcode_data1['hcode'] = hcode_data1['hcode'].astype(str)
hcode_data1['admdong_cd'] = [hcode_data1.loc[i, 'hcode'][:8] for i in hcode_data1.index]

#df4 = df4.loc[df4['admdong_cd'].str.contains('26470620')].copy()
df4 = pd.merge(df4, hcode_data1, "inner", "admdong_cd")

# 주단위 분류
df4.loc[(df4['base_month'].str.contains('09')) & (df4['base_day'].str.contains('19|20|21|22|23')), "week_num"] = "9월 3주중"
df4.loc[(df4['base_month'].str.contains('09')) & (df4['base_day'].str.contains('26|27|28|29|30')), "week_num"] = "9월 4주중"
df4.loc[(df4['base_month'].str.contains('09')) & (df4['base_day'].str.contains('17|18')), "week_num"] = "9월 2주말"
df4.loc[(df4['base_month'].str.contains('09')) & (df4['base_day'].str.contains('24|25')), "week_num"] = "9월 3주말"
df4.loc[(df4['base_month'].str.contains('10')) & (df4['base_day'].str.contains('1|2')), "week_num"] = "9월 4주말"
df4.loc[(df4['base_month'].str.contains('10')) & (df4['base_day'].str.contains('3|4|5|6|7')), "week_num"] = "10월 1주중"
df4.loc[(df4['base_month'].str.contains('10')) & (df4['base_day'].str.contains('8|9')), "week_num"] = "10월 1주말"
df4.loc[(df4['base_month'].str.contains('10')) & (df4['base_day'].str.contains('10|11|12|13|14')), "week_num"] = "10월 2주중"
df4.loc[(df4['base_month'].str.contains('10')) & (df4['base_day'].str.contains('15|16')), "week_num"] = "10월 2주말"
df4.loc[(df4['base_month'].str.contains('10')) & (df4['base_day'].str.contains('17|18|19|20')), "week_num"] = "10월 3주중"

# 지역 기준 국적별 장기체류 생활인구 수
# 동북아시아 : 대만, 몽골, 일본, 중국
df4_agg_eastnorth = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['twn','jpn','chn','hkg'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df4_agg_eastnorth.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','타이완','일본','중국','홍콩']
df4_agg_eastnorth.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_동북아시아.csv", sep="|", encoding="utf-8")

# 동남아시아 : 말레이시아, 미얀마, 베트남, 캄보디아, 태국, 필리핀, 인도네시아, 말레이시아
df4_agg_eastsouth = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['vnm','phl','idn','tha','mys'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df4_agg_eastsouth.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','베트남','필리핀','인도네시아','태국','말레이시아']
df4_agg_eastsouth.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_동남아시아.csv", sep="|", encoding="utf-8")

# 서남아시아 : 네팔, 스리랑카, 방글라데시, 파키스탄, 인도, 오스트레일리아
df4_agg_westsouth = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['ind','aus'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df4_agg_westsouth.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','인도','호주']
df4_agg_westsouth.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_서남아시아.csv", sep="|", encoding="utf-8")

# 북미 : 미국, 캐나다
df4_agg_northameria = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['usa','can'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df4_agg_northameria.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','미국','캐나다']
df4_agg_northameria.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_북미.csv", sep="|", encoding="utf-8")

# 러시아 중앙아시아 : 러시아, 카자흐스탄, 키르기스스탄, 유즈베키스탄
df4_agg_midasia = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['rus'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df4_agg_midasia.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','러시아']
df4_agg_midasia.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_러시아중앙아시아.csv", sep="|", encoding="utf-8")

# 유럽 : 독일, 영국, 프랑스, 우크라이나
df4_agg_europe = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['gbr','deu','fra'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df4_agg_europe.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','영국','독일','프랑스']
df4_agg_europe.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_러시아중앙아시아.csv", sep="|", encoding="utf-8")

# 중동북아프리카 : 이집트
df4_agg_mideast = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['gcc'].sum().reset_index().sort_values(by='admdong_cd')
df4_agg_mideast.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','걸프협력회의기구']
df4_agg_mideast.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_중동북아프리카.csv", sep="|", encoding="utf-8")

# 사하라 이남 아프리카와 기타 : 나이지리아, 기타
df4_agg_sahara = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['etc'].sum().round(0).reset_index().sort_values(by='admdong_cd')
df4_agg_sahara.columns=['기준일','주차','요일','시간대','행정동코드','행정동명','기타국가']
df4_agg_sahara.to_csv("./본청_대변인실/집계결과/주간_시간대_행정동별_외국인단기체류생활인구현황_사하라이남아프리카기타.csv", sep="|", encoding="utf-8")

print(" ========== 행정동별 단기체류 외국인 생활인구 현황집계 완료 ========== ")

 ========== 행정동별 단기체류 외국인 생활인구 현황집계 시작 ========== 


<ipython-input-52-ed7cd410edae>:36: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df4_agg_eastnorth = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['twn','jpn','chn','hkg'].sum().round(0).reset_index().sort_values(by='admdong_cd')
<ipython-input-52-ed7cd410edae>:41: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df4_agg_eastsouth = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['vnm','phl','idn','tha','mys'].sum().round(0).reset_index().sort_values(by='admdong_cd')
<ipython-input-52-ed7cd410edae>:46: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df4_agg_westsouth = df4.groupby(['date','week_num','요일','timezn_cd','admdong_cd','hname'])['ind','aus'].sum().round(0).reset_index().sort_values(by='admdong_

 ========== 행정동별 단기체류 외국인 생활인구 현황집계 완료 ========== 


#### 카드이용 현황분석

In [56]:

print(" ========== 행사기간 전후 행정동별 카드이용 현황집계 시작 ========== ")
# 내국인 카드이용 현황
df5 = pd.read_csv("./본청_대변인실/data/소비매출/busan_d_adm_amt_202209.txt", sep="|", encoding="utf-8")
df5['STD_YM'] = df5['STD_YM'].astype(str)
df5['STD_YMD'] = df5['STD_YMD'].astype(str)
df5['YEAR'] = [df5.loc[i, 'STD_YM'][:4] for i in df5.index]
df5['MONTH'] = [df5.loc[i, 'STD_YM'][4:] for i in df5.index]
df5['DAY'] = [df5.loc[i, 'STD_YMD'][6:] for i in df5.index]
df5['DATE'] = df5['YEAR'] + "-" + df5['MONTH'] + '-' + df5['DAY']
df5['DATE'] = pd.to_datetime(df5['DATE'])
df5['WEEKDAY'] = df5['DATE'].dt.weekday
df5['요일'] = df5['DATE'].dt.day_name()
df5['ADSTRD_CODE'] = df5['ADSTRD_CODE'].astype(str)
df5['AGRDE_CODE'] = df5['AGRDE_CODE'].astype(str)
df5['TMZON_CODE'] = df5['TMZON_CODE'].astype(str)

hcode_data1 = pd.read_csv("./공통코드/행정동코드.csv", sep=",", encoding="utf-8")
hcode_data1['hcode'] = hcode_data1['hcode'].astype(str)
hcode_data1.rename(columns={'hcode':'ADSTRD_CODE'}, inplace=True)

df5 = df5.loc[df5['hcode'].str.contains('2647062000')].copy()
df5 = pd.merge(df5, hcode_data1, 'inner', 'ADSTRD_CODE')
del df5['scode']
del df5['dcode']
del df5['sname']
del df5['do_name']

induty_data = pd.read_csv("./공통코드/업종분류코드.csv", sep=",", encoding="utf-8")
induty_data.columns=['INDUTY_SE_LCLAS','INDUTY_SE_LCLAS_NM','INDUTY_SE_MLSFC','INDUTY_SE_MLSFC_NM','INDUTY_SE_SCLAS','INDUTY_SE_SCLAS_NM']
df5 = pd.merge(df5, induty_data, 'inner', 'INDUTY_SE_MLSFC')

# 주단위 분류
df5.loc[(df5['MONTH'].str.contains('09')) & (df5['DAY'].str.contains('19|20|21|22|23')), "WEEK_NUM"] = "9월 3주중"
df5.loc[(df5['MONTH'].str.contains('09')) & (df5['DAY'].str.contains('26|27|28|29|30')), "WEEK_NUM"] = "9월 4주중"
df5.loc[(df5['MONTH'].str.contains('09')) & (df5['DAY'].str.contains('17|18')), "WEEK_NUM"] = "9월 2주중"
df5.loc[(df5['MONTH'].str.contains('09')) & (df5['DAY'].str.contains('24|25')), "WEEK_NUM"] = "9월 3주말"
df5.loc[(df5['MONTH'].str.contains('10')) & (df5['DAY'].str.contains('1|2')), "WEEK_NUM"] = "9월 4주말"
df5.loc[(df5['MONTH'].str.contains('10')) & (df5['DAY'].str.contains('3|4|5|6|7')), "WEEK_NUM"] = "10월 1주중"
df5.loc[(df5['MONTH'].str.contains('10')) & (df5['DAY'].str.contains('8|9')), "WEEK_NUM"] = "10월 1주말"
df5.loc[(df5['MONTH'].str.contains('10')) & (df5['DAY'].str.contains('10|11|12|13|14')), "WEEK_NUM"] = "10월 2주중"
df5.loc[(df5['MONTH'].str.contains('10')) & (df5['DAY'].str.contains('15|16')), "WEEK_NUM"] = "10월 2주말"
df5.loc[(df5['MONTH'].str.contains('10')) & (df5['DAY'].str.contains('17|18|19|20')), "WEEK_NUM"] = "10월 3주중"

# 성별
df5['SEXDSTN_CODE'] = df5['SEXDSTN_CODE'].fillna('C')
df5.loc[df5['SEXDSTN_CODE'].str.contains('M'), "성별"] = "남성"
df5.loc[df5['SEXDSTN_CODE'].str.contains('F'), "성별"] = "여성"
df5.loc[df5['SEXDSTN_CODE'].str.contains('C'), "성별"] = "법인"

# 연령
df5.loc[df5['AGRDE_CODE'].str.contains('1'), "연령"] = "10G"
df5.loc[df5['AGRDE_CODE'].str.contains('2'), "연령"] = "20G"
df5.loc[df5['AGRDE_CODE'].str.contains('3'), "연령"] = "30G"
df5.loc[df5['AGRDE_CODE'].str.contains('4'), "연령"] = "40G"
df5.loc[df5['AGRDE_CODE'].str.contains('5'), "연령"] = "50G"
df5.loc[df5['AGRDE_CODE'].str.contains('6'), "연령"] = "60G"
df5.loc[df5['AGRDE_CODE'].str.contains('7'), "연령"] = "70G"

# 행정동별 카드이용금액 변화 집계
df5_mf_agg = df5.groupby(['DATE','WEEK_NUM','요일','ADSTRD_CODE','hname','성별'])['CARD_USE_AMOUNT'].sum().reset_index().sort_values(by="ADSTRD_CODE")
df5_mf_agg.columns = ['기준일', '주차', '요일', '행정동코드', '행정동명', '성별', '카드이용금액']
df5_mf_agg.to_csv("./본청_대변인실/집계결과/행정동별_성별_이용금액현황.csv", sep="|", index=False, encoding="utf-8")

df5_age_agg = df5.groupby(['DATE','WEEK_NUM','요일','ADSTRD_CODE','hname','연령'])['CARD_USE_AMOUNT'].sum().reset_index().sort_values(by="ADSTRD_CODE")
df5_age_agg.columns = ['기준일', '주차', '요일', '행정동코드', '행정동명', '연령', '카드이용금액']
df5_age_agg.to_csv("./본청_대변인실/집계결과/행정동별_연령_이용금액현황.csv", sep="|", index=False, encoding="utf-8")

df5_knd_agg = df5.groupby(['DATE','WEEK_NUM','요일','ADSTRD_CODE','hname','INDUTY_SE_MLSFC_NM'])['CARD_USE_AMOUNT'].sum().reset_index().sort_values(by="ADSTRD_CODE")
df5_knd_agg.columns = ['기준일', '주차', '요일', '행정동코드', '행정동명', '업종중분류', '카드이용금액']
df5_knd_agg.to_csv("./본청_대변인실/집계결과/행정동별_업종_이용금액현황.csv", sep="|", index=False, encoding="utf-8")

print(" ========== 행사기간 전후 행정동별 카드이용 현황집계 완료 ========== ")


 ========== 행사기간 전후 행정동별 카드이용 현황집계 시작 ========== 
 ========== 행사기간 전후 행정동별 카드이용 현황집계 완료 ========== 


In [3]:
df6 = pd.read_csv("./본청_대변인실/data/소비매출/busan_d_tour_nlty_amt_220901-221020.txt", sep="|",  encoding="utf-8")
df6['STD_YM'].unique()

array([201910, 201909], dtype=int64)